# Hands-on 4: Subscribe Data MQTT dengan Python

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 3.

Tujuan:
- Terhubung ke broker MQTT sebagai subscriber.
- Berlangganan satu topik dan topik wildcard (`kampus/lab1/+/telemetry`).
- Menerima pesan real-time dan mengubah payload JSON menjadi objek Python.
- Melakukan validasi awal data (kelengkapan field dan rentang nilai).

Prasyarat: publisher Hands-on 3 (`scripts/mqtt/publish_sensor.py`) dan broker MQTT lokal.
Notebook ini menjalankan broker lokal berbasis Python (`scripts/mqtt/run_broker.py`) agar alur lengkap dapat dieksekusi tanpa instalasi sistem tambahan.

In [1]:
import os
from pathlib import Path

# Pastikan working directory adalah root repositori,
# karena nbconvert mengeksekusi notebook dari folder notebooks/.
while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


In [2]:
import json
import subprocess
import sys
import time
from pathlib import Path

PYTHON = sys.executable
BROKER = "127.0.0.1"
PORT = 1883
LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def start_process(args, stdout_file):
    f = open(stdout_file, "w", encoding="utf-8")
    proc = subprocess.Popen(args, stdout=f, stderr=subprocess.STDOUT)
    return proc, f

print("[INFO] Utilitas siap. Python:", PYTHON)

[INFO] Utilitas siap. Python: /home/sandbox/iot-ml-book/.venv/bin/python3


## 1. Menjalankan Broker MQTT Lokal

Buku memakai Mosquitto (`mosquitto -v`). Di sini kita pakai broker lokal berbasis Python
yang setara (`scripts/mqtt/run_broker.py`) pada `127.0.0.1:1883`.

In [3]:
broker_proc, broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_nb.log"
)
time.sleep(3)
print("[INFO] Broker PID:", broker_proc.pid)
print("[INFO] Broker berjalan:", broker_proc.poll() is None)

[INFO] Broker PID: 1318
[INFO] Broker berjalan: True


## 2. Menjalankan Publisher (Hands-on 3)

Tiga publisher mengirim telemetri ke `kampus/lab1/room01/telemetry`, `room02`, dan `room03`
agar subscriber wildcard dapat didemonstrasikan.

In [4]:
publishers = []
for room in ["room01", "room02", "room03"]:
    proc, f = start_process(
        [
            PYTHON, "scripts/mqtt/publish_sensor.py",
            "--topic", f"kampus/lab1/{room}/telemetry",
            "--count", "30", "--interval", "0.5",
        ],
        LOG_DIR / f"publisher_{room}_nb.log",
    )
    publishers.append((room, proc, f))
    print(f"[INFO] Publisher {room} PID: {proc.pid}")
time.sleep(1)
print("[INFO] Publisher berjalan.")

[INFO] Publisher room01 PID: 1319
[INFO] Publisher room02 PID: 1320
[INFO] Publisher room03 PID: 1321


[INFO] Publisher berjalan.


## 3. Subscriber Sederhana (Satu Topik)

`scripts/mqtt/subscribe_sensor.py` berlangganan `kampus/lab1/room01/telemetry`,
menampilkan payload mentah, dan memparsing field JSON.

In [5]:
hasil = subprocess.run(
    [
        PYTHON, "scripts/mqtt/subscribe_sensor.py",
        "--max-messages", "3", "--timeout", "25",
    ],
    capture_output=True, text=True, timeout=60,
)
print(hasil.stdout)
print(hasil.stderr)

[INFO] Terhubung ke broker MQTT.
[INFO] Subscribe ke topik: kampus/lab1/room01/telemetry

=== Pesan baru diterima ===
Topik   : kampus/lab1/room01/telemetry
QoS     : 1
Payload : {"device_id": "room01", "temperature_c": 27.03, "humidity_pct": 57.28, "co2_ppm": 785, "power_watt": 129.73, "timestamp": "2026-10-22T07:44:52"}
[SUKSES] Payload JSON valid.
Device ID     : room01
Temperature C : 27.03
Humidity %    : 57.28
CO2 ppm       : 785
Power watt    : 129.73
Timestamp     : 2026-10-22T07:44:52

=== Pesan baru diterima ===
Topik   : kampus/lab1/room01/telemetry
QoS     : 1
Payload : {"device_id": "room01", "temperature_c": 29.71, "humidity_pct": 50.51, "co2_ppm": 849, "power_watt": 120.34, "timestamp": "2026-10-22T07:44:53"}
[SUKSES] Payload JSON valid.
Device ID     : room01
Temperature C : 29.71
Humidity %    : 50.51
CO2 ppm       : 849
Power watt    : 120.34
Timestamp     : 2026-10-22T07:44:53

=== Pesan baru diterima ===
Topik   : kampus/lab1/room01/telemetry
QoS     : 1
Payload : {

## 4. Subscriber Wildcard untuk Banyak Perangkat

`scripts/mqtt/subscribe_all_rooms.py` memakai topik `kampus/lab1/+/telemetry`
sehingga membaca room01, room02, dan room03 sekaligus.

In [6]:
hasil = subprocess.run(
    [
        PYTHON, "scripts/mqtt/subscribe_all_rooms.py",
        "--max-messages", "6", "--timeout", "25",
    ],
    capture_output=True, text=True, timeout=60,
)
print(hasil.stdout)
print(hasil.stderr)

[INFO] Terhubung ke broker.
[INFO] Subscribe wildcard ke: kampus/lab1/+/telemetry
[kampus/lab1/room02/telemetry] {'device_id': 'room01', 'temperature_c': 28.17, 'humidity_pct': 51.8, 'co2_ppm': 900, 'power_watt': 141.36, 'timestamp': '2026-10-22T07:44:54'}
[kampus/lab1/room01/telemetry] {'device_id': 'room01', 'temperature_c': 24.63, 'humidity_pct': 52.34, 'co2_ppm': 477, 'power_watt': 115.84, 'timestamp': '2026-10-22T07:44:54'}
[kampus/lab1/room03/telemetry] {'device_id': 'room01', 'temperature_c': 28.55, 'humidity_pct': 47.52, 'co2_ppm': 884, 'power_watt': 153.79, 'timestamp': '2026-10-22T07:44:54'}
[kampus/lab1/room02/telemetry] {'device_id': 'room01', 'temperature_c': 25.69, 'humidity_pct': 55.35, 'co2_ppm': 694, 'power_watt': 110.88, 'timestamp': '2026-10-22T07:44:54'}
[kampus/lab1/room01/telemetry] {'device_id': 'room01', 'temperature_c': 27.89, 'humidity_pct': 64.38, 'co2_ppm': 806, 'power_watt': 131.89, 'timestamp': '2026-10-22T07:44:54'}
[kampus/lab1/room03/telemetry] {'device

## 5. Subscriber dengan Validasi Payload

`scripts/mqtt/subscribe_validated.py` memeriksa kelengkapan field dan rentang nilai:
`suhu 0-60 C`, `kelembaban 0-100 %`, `CO2 300-5000 ppm`.

Untuk mendemonstrasikan deteksi data tidak valid, subscriber dijalankan di latar
belakang, lalu dua payload rusak (suhu di luar rentang dan payload non-JSON) dikirim
ke topik yang sama di tengah arus data normal.

In [7]:
# Kirim payload tidak valid SAAT subscriber validasi berjalan.
import paho.mqtt.publish as mqttpub

def kirim_payload_rusak():
    payload_rusak = {
        "device_id": "room03",
        "temperature_c": 95.0,  # di luar rentang valid 0-60
        "humidity_pct": 50.0,
        "co2_ppm": 700,
        "power_watt": 120.0,
        "timestamp": "2026-10-22T00:00:00",
    }
    mqttpub.single(
        "kampus/lab1/room03/telemetry",
        json.dumps(payload_rusak),
        qos=1, hostname=BROKER, port=PORT,
        client_id="publisher-anomali",
    )
    mqttpub.single(
        "kampus/lab1/room03/telemetry",
        "bukan-json",
        qos=1, hostname=BROKER, port=PORT,
        client_id="publisher-anomali",
    )
    print("[INFO] 2 payload uji (tidak valid) dikirim.")

In [8]:
# Jalankan subscriber validasi di latar belakang,
# lalu kirim payload rusak di tengah arus data normal.
val_proc, val_f = start_process(
    [
        PYTHON, "scripts/mqtt/subscribe_validated.py",
        "--timeout", "10",
    ],
    LOG_DIR / "subscriber_validated_nb_run.log",
)
time.sleep(2)  # beri waktu subscriber tersambung dan subscribe
kirim_payload_rusak()
val_proc.wait(timeout=60)
val_f.close()
print((LOG_DIR / "subscriber_validated_nb_run.log").read_text(encoding="utf-8"))

[INFO] 2 payload uji (tidak valid) dikirim.


[INFO] Terhubung ke broker.
[INFO] Subscribe dengan validasi ke: kampus/lab1/+/telemetry
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 25.88, 'humidity_pct': 58.52, 'co2_ppm': 528, 'power_watt': 151.39, 'timestamp': '2026-10-22T07:44:55'}
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 25.5, 'humidity_pct': 56.3, 'co2_ppm': 585, 'power_watt': 118.89, 'timestamp': '2026-10-22T07:44:55'}
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 26.85, 'humidity_pct': 55.49, 'co2_ppm': 496, 'power_watt': 140.45, 'timestamp': '2026-10-22T07:44:55'}
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 26.61, 'humidity_pct': 54.23, 'co2_ppm': 894, 'power_watt': 112.01, 'timestamp': '2026-10-22T07:44:55'}
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 29.27, 'humidity_pct': 58.99, 'co2_ppm': 569, 'power_watt': 128.56, 'timestamp': '2026-10-22T07:44:55'}
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 28.24, 'humidity_pct': 59.6

## 6. Verifikasi Log dan Pembersihan

Semua subscriber menulis ringkasan ke `outputs/logs/`. Di akhir sesi,
publisher dan broker dihentikan.

In [9]:
for room, proc, f in publishers:
    proc.terminate()
    f.close()
broker_proc.terminate()
broker_f.close()
time.sleep(1)
print("[INFO] Publisher dan broker dihentikan.")

for log_name in ["subscriber_room01.log", "subscriber_all_rooms.log", "subscriber_validated.log"]:
    log_path = LOG_DIR / log_name
    print(f"\n===== {log_name} (5 baris terakhir) =====")
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8").strip().splitlines()
        for line in lines[-5:]:
            print(line)
    else:
        print("[PERINGATAN] Log belum ada.")

[INFO] Publisher dan broker dihentikan.

===== subscriber_room01.log (5 baris terakhir) =====
2026-10-22T07:44:34.429541 - Pesan valid dari kampus/lab1/room01/telemetry: {"device_id": "room01", "temperature_c": 27.21, "humidity_pct": 50.74, "co2_ppm": 719, "power_watt": 103.33, "timestamp": "2026-10-22T07:44:34"}
2026-10-22T07:44:52.652247 - Subscribe ke topik: kampus/lab1/room01/telemetry
2026-10-22T07:44:52.721620 - Pesan valid dari kampus/lab1/room01/telemetry: {"device_id": "room01", "temperature_c": 27.03, "humidity_pct": 57.28, "co2_ppm": 785, "power_watt": 129.73, "timestamp": "2026-10-22T07:44:52"}
2026-10-22T07:44:53.221803 - Pesan valid dari kampus/lab1/room01/telemetry: {"device_id": "room01", "temperature_c": 29.71, "humidity_pct": 50.51, "co2_ppm": 849, "power_watt": 120.34, "timestamp": "2026-10-22T07:44:53"}
2026-10-22T07:44:53.722145 - Pesan valid dari kampus/lab1/room01/telemetry: {"device_id": "room01", "temperature_c": 24.09, "humidity_pct": 51.25, "co2_ppm": 520, "p

## Penutup

Subscriber Python kini menjadi "mata digital" yang membaca arus telemetri MQTT secara
real-time, memvalidasi payload, dan menyiapkan data untuk tahap penyimpanan
(Hands-on 5: CSV / SQLite / InfluxDB).